## Mini Project (OpsPulse): Pick a dataset (NYC Taxi, Public API – weather, finance). Fetch data from a public API or dataset. Save raw data to disk using JSON/CSV, transform it using Pandas. Add 2-3 basic pytest tests.

In [1]:
!pip install requests

In [15]:
import requests
import json
import pandas as pd
from pathlib import Path

# -----------------------------------
# 1. Create folders
# -----------------------------------
raw_folder = Path("data/raw")
processed_folder = Path("data/processed")
test_folder = Path("tests")

raw_folder.mkdir(parents=True, exist_ok=True)
processed_folder.mkdir(parents=True, exist_ok=True)
test_folder.mkdir(parents=True, exist_ok=True)

# -----------------------------------
# 2. Fetch weather data from API
# -----------------------------------
url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 6.9271,
    "longitude": 79.8612,
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "wind_speed_10m"
    ],
    "timezone": "Asia/Colombo",
    "forecast_days": 3
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

weather_data = response.json()




In [17]:
with open("data/raw/weather_raw.json", "w", encoding="utf-8") as file:
    json.dump(weather_data, file, indent=4)

print("Weather data saved successfully!")

Weather data saved successfully!


In [21]:
import pandas as pd



df = pd.DataFrame(weather_data["hourly"])

print(df.head())

               time  temperature_2m  relative_humidity_2m  precipitation  \
0  2026-08-07T00:00            26.6                    84            0.2   
1  2026-08-07T01:00            26.7                    81            0.2   
2  2026-08-07T02:00            26.1                    88            0.3   
3  2026-08-07T03:00            26.5                    86            0.1   
4  2026-08-07T04:00            26.4                    82            0.1   

   wind_speed_10m  
0            17.0  
1            16.7  
2            15.9  
3            18.3  
4            19.5  


In [23]:


# Convert time to datetime
df["time"] = pd.to_datetime(df["time"])

# Rename columns
df = df.rename(columns={
    "temperature_2m": "temperature_c",
    "relative_humidity_2m": "humidity_pct",
    "wind_speed_10m": "wind_speed_kmh"
})



In [25]:
# Handle missing values
df = df.dropna()



In [27]:
# Add temperature category
df["temperature_status"] = df["temperature_c"].apply(
    lambda x: "Hot" if x >= 30
    else "Warm" if x >= 25
    else "Cool"
)

In [29]:
# Add raining indicator
df["is_raining"] = df["precipitation"].apply(
    lambda x: True if x > 0 else False
)


In [31]:
# Add date column
df["date"] = df["time"].dt.date


In [33]:
# Select final columns
final_df = df[
    [
        "time",
        "date",
        "temperature_c",
        "temperature_status",
        "humidity_pct",
        "precipitation",
        "is_raining",
        "wind_speed_kmh"
    ]
]

In [35]:
# -----------------------------------
#  Save transformed data as CSV
# -----------------------------------
final_df.to_csv(
    "data/processed/weather_clean.csv",
    index=False
)
print("Processed CSV saved successfully.")


In [37]:
# -----------------------------------
#  Display transformed data
# -----------------------------------
print("\nProcessed Data:")
print(final_df.head())

print("\nNumber of rows:", len(final_df))

print("\nMissing values:")
print(final_df.isnull().sum())


Processed Data:
                 time        date  temperature_c temperature_status  \
0 2026-08-07 00:00:00  2026-08-07           26.6               Warm   
1 2026-08-07 01:00:00  2026-08-07           26.7               Warm   
2 2026-08-07 02:00:00  2026-08-07           26.1               Warm   
3 2026-08-07 03:00:00  2026-08-07           26.5               Warm   
4 2026-08-07 04:00:00  2026-08-07           26.4               Warm   

   humidity_pct  precipitation  is_raining  wind_speed_kmh  
0            84            0.2        True            17.0  
1            81            0.2        True            16.7  
2            88            0.3        True            15.9  
3            86            0.1        True            18.3  
4            82            0.1        True            19.5  

Number of rows: 72

Missing values:
time                  0
date                  0
temperature_c         0
temperature_status    0
humidity_pct          0
precipitation         0
is_rainin

In [39]:
# -----------------------------------
# 8. Create pytest test file
# -----------------------------------
test_code = '''
import pandas as pd

DATA_FILE = "data/processed/weather_clean.csv"

def load_data():
    return pd.read_csv(DATA_FILE)

def test_columns_exist():
    df = load_data()

    required_columns = [
        "time",
        "date",
        "temperature_c",
        "temperature_status",
        "humidity_pct",
        "precipitation",
        "is_raining",
        "wind_speed_kmh"
    ]

    for column in required_columns:
        assert column in df.columns

def test_temperature_no_nulls():
    df = load_data()
    assert df["temperature_c"].isnull().sum() == 0

def test_precipitation_non_negative():
    df = load_data()
    assert (df["precipitation"] >= 0).all()
'''

with open(
    "tests/test_weather_pipeline.py",
    "w",
    encoding="utf-8"
) as file:
    file.write(test_code)

print("\nPytest file created successfully.")


Pytest file created successfully.
